# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import itertools
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score
import warnings

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
warnings.filterwarnings('ignore')

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [3]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df.head(1)

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,uid_user_16,uid_user_17,uid_user_18,uid_user_19,uid_user_2,uid_user_20,uid_user_21,uid_user_22,uid_user_23,uid_user_24,uid_user_25,uid_user_26,uid_user_27,uid_user_28,uid_user_29,uid_user_3,uid_user_30,uid_user_31,uid_user_4,uid_user_6,uid_user_7,uid_user_8,labname_code_rvw,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1686 entries, 0 to 1685
Data columns (total 43 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   numTrials         1686 non-null   int64  
 1   hour              1686 non-null   int64  
 2   uid_user_0        1686 non-null   float64
 3   uid_user_1        1686 non-null   float64
 4   uid_user_10       1686 non-null   float64
 5   uid_user_11       1686 non-null   float64
 6   uid_user_12       1686 non-null   float64
 7   uid_user_13       1686 non-null   float64
 8   uid_user_14       1686 non-null   float64
 9   uid_user_15       1686 non-null   float64
 10  uid_user_16       1686 non-null   float64
 11  uid_user_17       1686 non-null   float64
 12  uid_user_18       1686 non-null   float64
 13  uid_user_19       1686 non-null   float64
 14  uid_user_2        1686 non-null   float64
 15  uid_user_20       1686 non-null   float64
 16  uid_user_21       1686 non-null   float64


In [5]:
day_of_week = pd.read_csv('../data/dayofweek.csv', usecols=['dayofweek'])
day_of_week.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1686 entries, 0 to 1685
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   dayofweek  1686 non-null   int64
dtypes: int64(1)
memory usage: 13.3 KB


In [6]:
df = pd.concat([day_of_week, df], axis=1)
df.head(1)

,dayofweek,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,uid_user_16,uid_user_17,uid_user_18,uid_user_19,uid_user_2,uid_user_20,uid_user_21,uid_user_22,uid_user_23,uid_user_24,uid_user_25,uid_user_26,uid_user_27,uid_user_28,uid_user_29,uid_user_3,uid_user_30,uid_user_31,uid_user_4,uid_user_6,uid_user_7,uid_user_8,labname_code_rvw,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,4,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [7]:
RANDOM_STATE=21
X = df.drop(columns=['dayofweek'])
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [8]:
params_svc = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}

svc = SVC(random_state=RANDOM_STATE, probability=True)

grid_search_svc = GridSearchCV(
    svc,
    params_svc,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_svc.fit(X_train, y_train)

GridSearchCV(estimator=SVC(probability=True, random_state=21), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 1.5, 5, 10],
                         'class_weight': ['balanced', None],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid']},
             scoring='accuracy')

In [9]:
print(f'Лучшие параметры: {grid_search_svc.best_params_}')
print(f'Accuracy на кросс-валидации: {grid_search_svc.best_score_}')

Лучшие параметры: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Accuracy на кросс-валидации: 0.8761090458488228


In [10]:
results_svc = pd.DataFrame(grid_search_svc.cv_results_)
cols = ['param_kernel', 'param_C', 'param_gamma', 'param_class_weight', 'mean_test_score', 'std_test_score', 'rank_test_score']
results_svc = results_svc[cols].sort_values('rank_test_score')
results_svc

,param_kernel,param_C,param_gamma,param_class_weight,mean_test_score,std_test_score,rank_test_score
70,rbf,10,auto,None,0.876109,0.018419,1
64,rbf,10,auto,balanced,0.863500,0.010870,2
58,rbf,5,auto,None,0.816018,0.008116,3
52,rbf,5,auto,balanced,0.808608,0.021007,4
63,linear,10,auto,balanced,0.721052,0.034438,5
60,linear,10,scale,balanced,0.721052,0.034438,5
66,linear,10,scale,None,0.719587,0.017463,7
69,linear,10,auto,None,0.719587,0.017463,7
51,linear,5,auto,balanced,0.706234,0.031619,9
48,linear,5,scale,balanced,0.706234,0.031619,9


**Выводы**: разница между топ-1 и топ-2 моделью составляет всего 1%, причём топ-1 забрала модель без дополнительного баланса весов. Однако с топ-5, который занял kernel=linear, разрыв уже 15%, что существенно. Хуже всего показала себя сигмоида: большая часть моделей с данным кернелом - это случайное угадывание

## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [11]:
params_tree = {
    'max_depth': range(1, 50),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

tree = DecisionTreeClassifier(random_state=RANDOM_STATE)

grid_search_tree = GridSearchCV(
    tree,
    params_tree,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_tree.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': range(1, 50)},
             scoring='accuracy')

In [12]:
print(f'Лучшие параметры: {grid_search_tree.best_params_}')
print(f'Accuracy на кросс-валидации: {grid_search_tree.best_score_}')

Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21}
Accuracy на кросс-валидации: 0.873864794162192


In [13]:
results_tree = pd.DataFrame(grid_search_tree.cv_results_)
cols = ['param_max_depth', 'param_class_weight', 'param_criterion', 'mean_test_score', 'std_test_score', 'rank_test_score']
results_tree = results_tree[cols].sort_values(['rank_test_score', 'param_max_depth'])
results_tree

,param_max_depth,param_class_weight,param_criterion,mean_test_score,std_test_score,rank_test_score
69,21,balanced,gini,0.873865,0.025066,1
73,25,balanced,gini,0.873854,0.025018,2
70,22,balanced,gini,0.872378,0.025263,3
71,23,balanced,gini,0.872372,0.025179,4
75,27,balanced,gini,0.872372,0.025179,4
76,28,balanced,gini,0.872372,0.025179,4
77,29,balanced,gini,0.872372,0.025179,4
78,30,balanced,gini,0.872372,0.025179,4
79,31,balanced,gini,0.872372,0.025179,4
80,32,balanced,gini,0.872372,0.025179,4


**Выводы**: Разница между первым и вторым местом составляет сотые процента, хотя глубина лучшей модели на 4 меньше. Так же существуют группы моделей с одинаковым скором, но с разной глубиной дерева. Это следствие того, что дерево при комбинации class_weight и criterion достигает своей естественной максимальной глубины раньше, чем max_depth начинает реально ограничивать рост (то есть дерево останавливается само - все листья становятся чистыми или упираются в другие критерии остановки). Поэтому любое значение max_depth, большее этой фактической глубины, попросту не влияет на итоговое дерево — получается идентичная модель, посчитанная многократно

## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [14]:
params_forest = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': range(1, 50),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

forest = RandomForestClassifier(random_state=RANDOM_STATE)

grid_search_forest = GridSearchCV(
    forest,
    params_forest,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_forest.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': range(1, 50),
                         'n_estimators': [5, 10, 50, 100]},
             scoring='accuracy')

In [15]:
print(f'Лучшие параметры: {grid_search_forest.best_params_}')
print(f'Accuracy на кросс-валидации: {grid_search_forest.best_score_}')

Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100}
Accuracy на кросс-валидации: 0.9042929918766351


In [16]:
results_forest = pd.DataFrame(grid_search_forest.cv_results_)
cols = ['param_n_estimators', 'param_max_depth', 'param_class_weight', 'param_criterion', 'mean_test_score', 'std_test_score', 'rank_test_score']
results_forest = results_forest[cols].sort_values(['rank_test_score'])
results_forest

,param_n_estimators,param_max_depth,param_class_weight,param_criterion,mean_test_score,std_test_score,rank_test_score
95,100,24,balanced,entropy,0.904293,0.012361,1
115,100,29,balanced,entropy,0.904290,0.012156,2
698,50,28,None,gini,0.904290,0.010961,2
314,50,30,balanced,gini,0.903549,0.012056,4
711,100,31,None,gini,0.903547,0.014380,5
99,100,25,balanced,entropy,0.902809,0.013639,6
326,50,33,balanced,gini,0.902809,0.013628,7
767,100,45,None,gini,0.902806,0.010460,8
779,100,48,None,gini,0.902806,0.010460,8
775,100,47,None,gini,0.902806,0.010460,8


**Выводы**: Лучшая комбинация - n_estimators=100, max_depth=24, class_weight='balanced', criterion='entropy' (accuracy примерно 0.904), но разница между топовыми моделями минимальна (~0.003) - можно смело брать более простую модель без потери качества. Критично важна только глубина: при max_depth <= 5 accuracy резко падает, независимо от остальных параметров. class_weight и criterion почти не влияют на результат

## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [17]:
keys = list(params_forest.keys())
values = list(params_forest.values())

combinations = list(itertools.product(*values))
results = []

for combo in tqdm(combinations):
    params = dict(zip(keys, combo))
    model = RandomForestClassifier(
        **params,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        n_jobs=-1
    )
    
    results.append({
        **params,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std()
    })

  0%|          | 0/784 [00:00<?, ?it/s]

In [18]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='mean_accuracy', ascending=False)
df_results

,n_estimators,max_depth,class_weight,criterion,mean_accuracy,std_accuracy
680,100,24,balanced,entropy,0.904293,0.012361
503,50,28,None,gini,0.904290,0.010961
700,100,29,balanced,entropy,0.904290,0.012156
509,50,30,balanced,gini,0.903549,0.012056
711,100,31,None,gini,0.903547,0.014380
684,100,25,balanced,entropy,0.902809,0.013639
521,50,33,balanced,gini,0.902809,0.013628
783,100,49,None,gini,0.902806,0.010460
507,50,29,None,gini,0.902806,0.011698
731,100,36,None,gini,0.902806,0.010460


**Выводы**: результаты аналогичны разделу 4

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [19]:
best_model = grid_search_forest.best_estimator_
y_pred = best_model.predict(X_test)
print(f'Accuracy на тестовой выборке: {accuracy_score(y_test, y_pred)}')

Accuracy на тестовой выборке: 0.9260355029585798
